# NLP Prototype — شرح الكود خطوة بخطوة

الـ notebook ده بيمشي على الـ pipeline اللي بيحوّل رسالة زي **"انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه"** لمعاملة مسجلة.

الـ pipeline من 5 مراحل:
1. **توحيد النص**: `nlp_proto/normalize.py`، قواعد
2. **المبلغ**: `nlp_proto/amounts.py`، قواعد (مش موديل)
3. **النوع والفئة**: `nlp_proto/tfidf.py`، TF-IDF + Logistic Regression، متدرب من الصفر
4. **الحاجة والمحل والتاريخ**: `nlp_proto/spacy_model.py`، spaCy NER، متدرب من الصفر
5. **الرد**: `nlp_proto/pipeline.py`، templates ثابتة بالمصري

> مفيش fine-tuning لموديل جاهز ومفيش LLM. Gemini استخدمناه بس عشان يكتب داتا التدريب.

## 0) الداتا

In [1]:
import pandas as pd
from nlp_proto.data import load_synthetic, split, load_handwritten

synthetic = load_synthetic()
train, test = split(synthetic)          # 80% تدريب / 20% اختبار، ثابت (seed=42)
handwritten = load_handwritten()        # 80 جملة مكتوبة يدوي للاختبار بس
print(f"synthetic: {len(synthetic)}  train: {len(train)}  test: {len(test)}  hand-written: {len(handwritten)}")
pd.DataFrame(synthetic)[["text", "intent", "amount", "category"]].sample(8, random_state=1)

synthetic: 2000  train: 1599  test: 401  hand-written: 80


,text,intent,amount,category
674,اشتراك يوتيوب بريميوم العائلي ب 175 جنيه,log_expense,175.0,Subscriptions
1699,What are my upcoming bill payments for next week?,question,NaN,NaN
1282,دفعت 780 جنيه قسط الأجهزة في بي تك,log_expense,780.0,Debt
1315,دفعت القسط رقم 6 من قرض البنك ب 3400 جنيه,log_expense,3400.0,Debt
1210,دفعت سهمين في الجمعية مع زمايلي ب 6000 جنيه,log_savings,6000.0,Savings
1636,معايا ١٠٠ ألف جنيه أحطهم في صندوق استثمار ولا ...,question,NaN,NaN
613,عمولة السمسار لما أجرنا الشقة كانت ألفين ونص,log_expense,2500.0,Housing
447,جبت دفاية زيت في الشتا ب ٣١٠٠ جنيه,log_expense,3100.0,Shopping


In [2]:
pd.Series([r["intent"] for r in synthetic]).value_counts()

log_expense     1243
log_income       289
other            129
log_transfer     108
question         104
log_savings       82
greeting          45
Name: count, dtype: int64

## 1) توحيد النص — `normalize.py`
بيحوّل الأرقام العربي لإنجليزي ويوحّد الحروف، عشان الموديل يشوف "العربية" و"العربيه" نفس الحاجة.

In [3]:
from nlp_proto.normalize import normalize
normalize("دفعت ١٢٠٠ جنيه للعربية في إسكندرية")

'دفعت 1200 جنيه للعربيه في اسكندريه'

## 2) المبلغ — `amounts.py` (قواعد، مش موديل)
بيلاقي كل الأرقام، ويدّي نقط للرقم اللي جنبه علامة ("ب" قبله أو "جنيه" بعده)، ويقلل نقط الرقم اللي جنبه "رصيد" أو وحدة زي "كيلو". وكمان بيفهم الأرقام بالكلام المصري.

In [4]:
from nlp_proto.amounts import extract_amount

examples = ["انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه", "دفعت ميتين جنيه بنزين",
            "دفعت مصاريف الكورس ألف وخمسمية", "طيرت باكو ونص في لبس شتوي",
            "اشتريت 3 تيشيرتات ب 900", "masarif el gam3a 9.5k",
            "Card ****4821 charged EGP 285.00 at TALABAT. Avl bal EGP 9,420.50"]
pd.DataFrame([{"text": t, **{k: v for k, v in extract_amount(t).items() if k != "span"}} for t in examples])

,text,amount,currency
0,انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه,1200.0,EGP
1,دفعت ميتين جنيه بنزين,200.0,EGP
2,دفعت مصاريف الكورس ألف وخمسمية,1500.0,EGP
3,طيرت باكو ونص في لبس شتوي,1500.0,EGP
4,اشتريت 3 تيشيرتات ب 900,900.0,EGP
5,masarif el gam3a 9.5k,9500.0,EGP
6,Card ****4821 charged EGP 285.00 at TALABAT. A...,285.0,EGP


In [5]:
# الدقة على كل رسايل الداتا اللي فيها مبلغ
tx = [r for r in synthetic if r["amount"] is not None]
ok = sum(extract_amount(r["text"])["amount"] == r["amount"] for r in tx)
print(f"amount accuracy: {ok}/{len(tx)} = {ok / len(tx):.1%}")

amount accuracy: 1591/1591 = 100.0%


## 3) النوع والفئة — TF-IDF + Logistic Regression (`tfidf.py`)

- **TF-IDF**: بيقطّع الرسالة لقطع حروف من 2 لـ 5 (char n-grams) ويحسب أهمية كل قطعة. **مش موديل بيتعلم.**
- **Logistic Regression**: موديل إحصائي بيتعلم أوزان: قطعة زي "طلبات" بتشاور على أكل.

الاتنين **متدربين من الصفر** على الـ 1,599 رسالة تدريب. (هنا بندرّب نسخة جوه الـ notebook من غير ما نلمس الموديلات المحفوظة في `models/`.)

In [6]:
from nlp_proto.tfidf import _model
from nlp_proto.data import TX_INTENTS

intent_model = _model().fit([normalize(r["text"]) for r in train], [r["intent"] for r in train])
tx_train = [r for r in train if r["intent"] in TX_INTENTS and r["category"]]
category_model = _model().fit([normalize(r["text"]) for r in tx_train], [f"{r['intent']}|{r['category']}" for r in tx_train])

vec = intent_model.named_steps["tfidfvectorizer"]
print("عدد القطع (features):", len(vec.vocabulary_))
print("أول قطع في رسالة:", vec.build_analyzer()(normalize("طلبت بيتزا"))[:15])

عدد القطع (features): 32717
أول قطع في رسالة: [' ط', 'طل', 'لب', 'بت', 'ت ', ' طل', 'طلب', 'لبت', 'بت ', ' طلب', 'طلبت', 'لبت ', ' طلبت', 'طلبت ', ' ب']


In [7]:
# الدقة على الاختبار
from sklearn.metrics import accuracy_score
print("intent accuracy:", f"{accuracy_score([r['intent'] for r in test], intent_model.predict([normalize(r['text']) for r in test])):.1%}")
tx_test = [r for r in test if r["intent"] in TX_INTENTS and r["category"]]
print("category accuracy (given correct intent):", f"{accuracy_score([f"{r['intent']}|{r['category']}" for r in tx_test], category_model.predict([normalize(r['text']) for r in tx_test])):.1%}")

intent accuracy: 87.3%
category accuracy (given correct intent): 79.3%


In [8]:
# الموديل اتعلم إيه؟ أهم قطع الحروف لكل فئة
import numpy as np
lr = category_model.named_steps["logisticregression"]
names = np.array(category_model.named_steps["tfidfvectorizer"].get_feature_names_out())
rows = []
for i, label in enumerate(lr.classes_):
    if label.startswith("log_expense|"):
        top = names[np.argsort(lr.coef_[i])[-8:][::-1]]
        rows.append({"category": label.split("|")[1], "top character n-grams": " · ".join(t.strip() for t in top)})
pd.DataFrame(rows)

,category,top character n-grams
0,Debt,قسط · قسط · سط · قس · سط · قسط · قسط · قس
1,Education,رس · رس · درس · rs · كورس · كتب · كورس · ورس
2,Entertainment,في · في · في · تذ · تذ · مع · ف · تذا
3,Food,و · 5 · من · ur · من · من · 2 · من
4,Health,شف · كشف · كشف · كشف · كشف · شف · كش · كش
5,Housing,شق · شقه · شقه · الشق · الشقه · الشق · لشقه · لشق
6,Shopping,من · من · من · ن · من · ب · جبت · جبت
7,Subscriptions,تراك · راك · شترا · شتراك · اشترا · تراك · راك...
8,Transport,لل · لل · lel · lel · lel · lel · le · ل
9,Utilities,نت · نت · شحن · شح · شحن · شح · حن · et


## 4) الحاجة والمحل والتاريخ — spaCy NER (`spacy_model.py`)

بدأنا بـ `spacy.blank("ar")`، يعني موديل **فاضي تماماً** (بيقطّع الكلام بس)، وضفناله طبقة NER (شبكة CNN صغيرة) اتدربت **من الصفر** على داتتنا.
التدريب بياخد حوالي 3 دقايق، فهنا بنحمّل الموديل المحفوظ من `python -m nlp_proto.train`. ولو عاوز تدرّب من جوه الـ notebook، خلي `TRAIN_SPACY = True`.

In [9]:
from nlp_proto.spacy_model import SpacyApproach

TRAIN_SPACY = False
ner = SpacyApproach()
if TRAIN_SPACY:
    ner.train(synthetic)
else:
    ner.load()
print(ner.main.pipe_names, "|", ner.main.get_pipe("ner").labels)

['ner', 'textcat'] | ('AMOUNT', 'CURRENCY', 'DATE', 'ITEM', 'MERCHANT', 'PERSON')


In [10]:
for t in ["انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه", "طلبت بيتزا من طلبات امبارح ب 320",
          "انا شحنت العربيه في محطة شحن الجيزة ب 350 جنيه", "Bought a jacket from Zara for 1500"]:
    doc = ner.main(t)
    print(t, "\n   →", [(e.text, e.label_) for e in doc.ents])

انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه 
   → [('كوتشي نايك', 'ITEM'), ('مول مصر', 'MERCHANT'), ('1200', 'AMOUNT'), ('جنيه', 'CURRENCY')]
طلبت بيتزا من طلبات امبارح ب 320 
   → [('طلبات', 'MERCHANT'), ('320', 'AMOUNT')]
انا شحنت العربيه في محطة شحن الجيزة ب 350 جنيه 
   → [('شحن الجيزة', 'ITEM'), ('350', 'AMOUNT'), ('جنيه', 'CURRENCY')]
Bought a jacket from Zara for 1500 
   → [('jacket', 'ITEM'), ('Zara', 'MERCHANT'), ('1500', 'AMOUNT')]


ملاحظة: في رسالة "محطة شحن الجيزة" الـ NER بيغلط، لأن تركيبة "في + مكان" مش متكررة كفاية في الداتا. ده من حدود موديل متدرب من الصفر على 2000 رسالة بس.

## 5) الـ pipeline كامل + الرد — `pipeline.py`

In [11]:
from nlp_proto.pipeline import Assistant
assistant = Assistant()
rows = []
for t in ["انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه", "اشتريت جزمة ب 200 جنيه من محل ابو جزم",
          "قبضت المرتب ١٥٠٠٠", "اشتريت جاكيت جديد", "الحلاق ١٥٠", "صرفت كام على الأكل الشهر ده؟"]:
    out = assistant.handle(t)
    d = out["extracted_data"]
    rows.append({"message": t, "status": out["status"], "amount": d.get("amount"), "category": d.get("category"),
                 "conf": d.get("category_confidence"), "item": d.get("item"), "merchant": d.get("merchant"), "reply": out["reply"]})
pd.DataFrame(rows)

,message,status,amount,category,conf,item,merchant,reply
0,انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه,confirmed,1200.0,Shopping,0.949,كوتشي نايك,مول مصر,"اتسجل ✅ مصروف: 1,200 جنيه – مشتريات (كوتشي ناي..."
1,اشتريت جزمة ب 200 جنيه من محل ابو جزم,confirmed,200.0,Shopping,0.829,جزمة,ابو جزم,اتسجل ✅ مصروف: 200 جنيه – مشتريات (جزمة من ابو...
2,قبضت المرتب ١٥٠٠٠,confirmed,15000.0,Salary,0.994,المرتب,NaN,"اتسجل ✅ دخل: 15,000 جنيه – مرتب (المرتب)"
3,اشتريت جاكيت جديد,needs_info,NaN,Shopping,0.779,جاكيت جديد,NaN,تمام، جاكيت جديد كان بكام؟
4,الحلاق ١٥٠,confirmed,150.0,Other,0.180,NaN,NaN,اتسجل ✅ مصروف: 150 جنيه – أخرى
5,صرفت كام على الأكل الشهر ده؟,not_applicable,NaN,NaN,NaN,NaN,NaN,الأسئلة عن مصاريفك جاية في النسخة الجاية 🙏 دلو...


## 6) مقارنة الطرق الثلاثة
النتايج دي من `python -m nlp_proto.evaluate` (محفوظة في `results/results.md`).

In [12]:
from IPython.display import Markdown
Markdown(open("results/results.md", encoding="utf-8").read())

Train: 1599 synthetic · Test: 401 synthetic (held out) + 80 hand-written

| Approach | Set | Type | Intent | Amount | Category | End-to-end |
|---|---|---|---|---|---|---|
| A. Rules (keywords + regex) | synthetic test | 78.3% | 76.8% | 100.0% | 44.7% | **46.4%** |
| | hand-written | 92.5% | – | 100.0% | 86.8% | **83.8%** |
| B. TF-IDF + Logistic Regression | synthetic test | 88.0% | 87.3% | 100.0% | 79.6% | **76.8%** |
| | hand-written | 100.0% | – | 100.0% | 92.1% | **92.5%** |
| C. spaCy (NER + textcat) | synthetic test | 88.8% | 88.0% | 99.7% | 71.1% | **72.8%** |
| | hand-written | 97.5% | – | 100.0% | 80.3% | **81.2%** |

| Approach | Model size | Latency / message | Training time |
|---|---|---|---|
| A. Rules (keywords + regex) | 0.0 MB | 0.04 ms | 0.0 s |
| B. TF-IDF + Logistic Regression | 7.23 MB | 0.88 ms | 4.5 s |
| C. spaCy (NER + textcat) | 34.72 MB | 3.31 ms | 146.7 s |

spaCy NER on synthetic test (exact span match): precision 84.2%, recall 79.6%, F1 **81.8%**. Entities skipped in training because of token misalignment: 98.


## 7) جرّب رسالتك

In [13]:
assistant.handle("اكتب رسالتك هنا ب 100 جنيه")

{'status': 'confirmed',
 'extracted_data': {'intent': np.str_('log_expense'),
  'type': 'expense',
  'amount': 100.0,
  'currency': 'EGP',
  'category': 'Education',
  'category_confidence': np.float64(0.269),
  'item': None,
  'merchant': None,
  'date': None,
  'person': None,
  'missing': []},
 'reply': 'اتسجل ✅ مصروف: 100 جنيه – تعليم'}